In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GeM(nn.Module):
  """Обобщенное среднее пулинг (Generalized Mean Pooling).

  Обучаемый аналог Average/Max Pooling.
  """

  def __init__(self, p=3.0, eps=1e-6):
    super().__init__()
    self.p = nn.Parameter(torch.tensor(p, dtype=torch.float32))
    self.eps = eps

  def forward(self, x):
    # Применяем степень p, берем среднее по пространству и извлекаем корень 1/p
    return F.avg_pool2d(
        x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))
    ).pow(1.0 / self.p)


class SEBlock(nn.Module):
  """Squeeze-and-Excitation блок для механизма внимания по каналам."""

  def __init__(self, channels, reduction=16):
    super().__init__()
    self.fc = nn.Sequential(
        nn.AdaptiveAvgPool2d(
            1
        ),  # Сжимаем пространственные размеры [B, C, H, W] -> [B, C, 1, 1]
        nn.Flatten(),
        nn.Linear(
            channels, channels // reduction, bias=False
        ),  # Уменьшаем размерность
        nn.ReLU(inplace=True),
        nn.Linear(
            channels // reduction, channels, bias=False
        ),  # Возвращаем исходное число каналов
        nn.Sigmoid(),  # Масштабирующие коэффициенты от 0 до 1
    )

  def forward(self, x):
    b, c, _, _ = x.shape
    w = self.fc(x).view(b, c, 1, 1)  # Получаем веса для каждого канала
    return x * w  # Умножаем входные карты признаков на веса внимания


class ResConvBlock(nn.Module):
  """Остаточный блок сверток с SE-вниманием и поддержкой изменения размерности."""

  def xxxx(self):  # заглушка для структуры
    pass

  def __init__(self, in_f, out_f, stride=1):
    super().__init__()
    self.conv1 = nn.Conv2d(
        in_f, out_f, kernel_size=3, stride=stride, padding=1, bias=False
    )
    self.bn1 = nn.BatchNorm2d(out_f)
    self.act = nn.SiLU()

    self.conv2 = nn.Conv2d(
        out_f, out_f, kernel_size=3, stride=1, padding=1, bias=False
    )
    self.bn2 = nn.BatchNorm2d(out_f)

    self.se = SEBlock(out_f, reduction=16)

    # Проекционный шорткат для выравнивания размеров, если меняются каналы или stride
    self.shortcut = nn.Sequential()
    if stride != 1 or in_f != out_f:
      self.shortcut = nn.Sequential(
          nn.Conv2d(
              in_f, out_f, kernel_size=1, stride=stride, bias=False
          ),  # 1x1 свертка для каналов
          nn.BatchNorm2d(out_f),
      )

  def forward(self, x):
    residual = self.shortcut(x)  # Сохраняем вход для остаточной связи

    out = self.conv1(x)
    out = self.bn1(out)
    out = self.act(out)

    out = self.conv2(out)
    out = self.bn2(out)
    out = self.se(out)  # Взвешиваем каналы по важности

    out += residual  # Складываем основной путь с остаточным (skip-connection)
    out = self.act(out)
    return out


class ProModernDigitCNN(nn.Module):

  def __init__(self, in_features: int = 1, num_classes: int = 10):
    super().__init__()

    # Начальный стем (превращает 1 канал в 32)
    self.stem = nn.Sequential(
        nn.Conv2d(in_features, 32, kernel_size=3, stride=1, padding=1, bias=False),
        nn.BatchNorm2d(32),
        nn.SiLU(),
    )

    # Иерархия остаточных блоков с уменьшением пространственного разрешения (stride=2)
    self.features = nn.Sequential(
        ResConvBlock(32, 64, stride=2),  # 28x28 -> 14x14
        ResConvBlock(64, 128, stride=2),  # 14x14 -> 7x7
        ResConvBlock(128, 256, stride=2),  # 7x7 -> 3x3
    )

    # Классификатор с GeM пулингом вместо жесткого AdaptiveAvgPool
    self.classifier = nn.Sequential(
        GeM(),
        nn.Flatten(),
        nn.Dropout(0.3),
        nn.Linear(256, num_classes),
    )

  def forward(self, x):
    x = self.stem(x)
    x = self.features(x)
    x = self.classifier(x)
    return x